# 🤖 AutoML Pilot - Training Template
Use this Colab notebook to run advanced AutoML (PyCaret) on your dataset. This template is designed to work seamlessly with datasets exported from the AutoML Pilot web app.

In [ ]:
# @title 1. Install Dependencies
# Install PyCaret and other necessary libraries
!pip install -q pycaret[full] ydata-profiling pandas
print('✅ Dependencies installed!')

In [ ]:
# @title 2. Load Dataset
import pandas as pd
from google.colab import files
import io

uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f'✅ Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns')
display(df.head())

In [ ]:
# @title 3. AutoML Setup & Training
from pycaret.classification import setup as cls_setup, compare_models as cls_compare, pull as cls_pull, save_model as cls_save, finalize_model as cls_finalize
from pycaret.regression import setup as reg_setup, compare_models as reg_compare, pull as reg_pull, save_model as reg_save, finalize_model as reg_finalize

# Configuration
target_column = 'target' # @param {type:"string"}
task_type = 'classification' # @param ["classification", "regression"]

print(f'🚀 Starting AutoML for {task_type} on target: {target_column}')

if task_type == 'classification':
    s = cls_setup(data=df, target=target_column, session_id=123, verbose=False, html=False)
    print('Comparing models...')
    best_model = cls_compare()
    leaderboard = cls_pull()
    display(leaderboard)
    
    # Finalize and Save
    final_model = cls_finalize(best_model)
    cls_save(final_model, 'best_model')
    print('✅ Classification model saved as best_model.pkl')

else:
    s = reg_setup(data=df, target=target_column, session_id=123, verbose=False, html=False)
    print('Comparing models...')
    best_model = reg_compare()
    leaderboard = reg_pull()
    display(leaderboard)
    
    # Finalize and Save
    final_model = reg_finalize(best_model)
    reg_save(final_model, 'best_model')
    print('✅ Regression model saved as best_model.pkl')

In [ ]:
# @title 4. Email Report (Optional)
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import json

send_email = False # @param {type:"boolean"}
email_address = "" # @param {type:"string"}
app_password = "" # @param {type:"string"}
recipient_email = "" # @param {type:"string"}

if send_email and email_address and app_password and recipient_email:
    try:
        msg = MIMEMultipart()
        msg['Subject'] = f"AutoML Report - {task_type.capitalize()}"
        msg['From'] = email_address
        msg['To'] = recipient_email
        
        metrics_html = leaderboard.to_html(classes='table table-striped')
        
        body = f"""
        <html>
        <body>
            <h2>AutoML Training Report</h2>
            <p>Task: {task_type}</p>
            <p>Target: {target_column}</p>
            <h3>Leaderboard</h3>
            {metrics_html}
        </body>
        </html>
        """
        msg.attach(MIMEText(body, 'html'))
        
        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
            server.login(email_address, app_password)
            server.send_message(msg)
        print("✅ Email report sent!")
    except Exception as e:
        print(f"❌ Failed to send email: {e}")

In [ ]:
# @title 5. Download Model
from google.colab import files
import os

if os.path.exists('best_model.pkl'):
    files.download('best_model.pkl')
    print('✅ Triggered download for best_model.pkl')
else:
    print('❌ Model file not found.')